# Data Engineering Mini-Project 1
## Student Success & Early-Warning Analytics System — Raw-to-Production Data Pipeline

**Name:** *(your name)*
**Course:** INS-605
**Date:** *(date)*
**Dataset used:** `university_student_success_raw_2500x25.csv` (or `.xlsx`)

**Goal:** Transform the raw, messy operational dataset into a clean, reliable, production-ready dataset following the 15-step data engineering pipeline, so it can later be used to train a machine learning model for academic risk prediction.

> Action item: Fill in your name/date above, and keep every heading in this notebook — the rubric awards 8 points specifically for proper headings and clear, meaningful output. Run every cell so outputs are visible before exporting to PDF.


## 1. Data Ingestion

**Action items:**
- Mount/connect Google Colab to your file (upload the file, or mount Google Drive).
- Load the dataset into a pandas DataFrame using `pd.read_csv()` or `pd.read_excel()`.
- Load the **data dictionary** file as well — you'll need it constantly to check expected data types and valid value ranges for every column.
- Print `df.shape` and `df.head()` to confirm the data loaded correctly (2,500 rows × 25 columns expected).


In [2]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [3]:
raw_path = "/content/drive/MyDrive/Could_Ml_classwork/project 1/university_student_success_raw_2500x25.csv"
dict_path = "/content/drive/MyDrive/Could_Ml_classwork/project 1/university_student_success_data_dictionary.xlsx"

df = pd.read_csv(raw_path)
data_dict = pd.read_excel(dict_path)

print("Shape:", df.shape)
df.head()

Shape: (2500, 25)


,Student_ID,Admission_Year,Age,Gender,Program,Study_Year,Semester,Attendance_Percent,Study_Hours_Per_Week,LMS_Logins_Per_Week,...,Credits_Registered,Credits_Earned,Failed_Courses,Backlog_Courses,Financial_Aid_Status,Scholarship_Amount,Library_Visits_Per_Month,Counseling_Visits,Extracurricular_Level,Academic_Risk_Level
0,20260617.0,2023.0,19,Male,NaN,2.0,4.0,87.2,7.6,NaN,...,16,16.0,0.0,0.0,Full,2500.0,4.0,4.0,Low,Low
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20261481.0,2025.0,24,Other,Electrical Engineering,3.0,6.0,74.8,13.3,10.0,...,16,13.0,1.0,1.0,NaN,NaN,3.0,2.0,Low,Medium
3,20260538.0,2023.0,24,Male,Civil Engineering,4.0,8.0,79.3,12.0,6.0,...,24,21.0,0.0,0.0,NaN,NaN,NaN,0.0,Low,Low
4,20261127.0,2024.0,22,NaN,Electrical Engineering,3.0,5.0,89.5,22.2,8.0,...,16,16.0,0.0,1.0,Partial,NaN,4.0,0.0,Low,Low


## 2. Data Profiling

**Action items:**
- Explore `df.info()` to see dtypes and non-null counts for all 25 columns.
- Run `df.describe(include='all')` to see basic statistics for numeric and categorical columns.
- List all column names and, using the data dictionary, classify each one as **Nominal**, **Ordinal**, **Discrete**, or **Continuous**. Write this classification out clearly in a markdown table or printed dictionary — this is explicitly graded.
- Check `df.nunique()` for each column to spot categorical columns with messy/inconsistent categories (e.g. `" male "`, `"FEMALE"`).
- Write a short paragraph (in markdown) summarizing the scenario, the columns, and the data-quality issues you can already see.


In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


In [ ]:
# Explore unique values per column to spot messy categories, typos, whitespace, etc.
for col in df.columns:
    print(col, "->", df[col].nunique(), "unique values")


**Variable classification** *(fill in based on the data dictionary)*

| Column | Type (Nominal / Ordinal / Discrete / Continuous) |
|---|---|
| *(e.g. Gender)* | Nominal |
| *(e.g. Academic_Risk_Level)* | Ordinal |
| *(e.g. Failed_Courses)* | Discrete |
| *(e.g. Current_GPA)* | Continuous |
| ... | ... |

**Summary of data-quality issues observed:** *(write 3–5 sentences here)*


## 3. Schema Validation

**Action items:**
- Compare each column's actual pandas dtype against the expected dtype in the data dictionary.
- Flag columns where numeric fields contain text (e.g. `Age = "twenty"`, `Current_GPA = "three"`, `Attendance_Percent = "eighty"`, `Credits_Registered = "eighteen"`).
- Do NOT fix these yet — just detect and document them here. The actual correction happens in the Data-Type Validation step later.


In [ ]:
# Detect columns that should be numeric but contain non-numeric strings
for col in df.select_dtypes(include='object').columns:
    non_numeric_like = df[col].apply(lambda x: isinstance(x, str) and x.strip().isalpha())
    if non_numeric_like.any():
        print(col, "->", df.loc[non_numeric_like, col].unique()[:10])


## 4. Duplicate Detection

**Action items:**
- Count exact duplicate rows with `df.duplicated().sum()`.
- Display a few duplicate rows to confirm they are truly identical.
- Remove them with `df.drop_duplicates()` and confirm the new shape.


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(by=df.columns[0]).head()


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)


## 5. Blank / Invalid Record Removal

**Action items:**
- Find completely blank rows (every column is NaN) with `df.isnull().all(axis=1)`.
- Drop them with `df.dropna(how='all')`.
- Confirm the new row count.


In [ ]:
blank_rows = df.isnull().all(axis=1).sum()
print("Completely blank rows:", blank_rows)

df = df.dropna(how='all').reset_index(drop=True)
print("Shape after removing blank rows:", df.shape)


## 6. Missing-Value Analysis

**Action items:**
- Calculate missing-value percentage per column: `df.isnull().mean() * 100`.
- Sort and display so the highest-missing columns are visible.
- Optionally visualize with a bar chart or heatmap (`seaborn.heatmap(df.isnull())`).
- Note which columns have Nominal / Ordinal / Discrete / Continuous missing values — you'll treat each differently later.


In [ ]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_pct


In [ ]:
import matplotlib.pyplot as plt
missing_pct[missing_pct > 0].plot(kind='barh', figsize=(8,6))
plt.xlabel("% Missing")
plt.title("Missing Values by Column")
plt.show()


## 7. Remove / Retain High-Missing Columns

**Action items:**
- Identify any column with **>70% missing values** — the rubric expects you to correctly find and remove it.
- Drop that column and explain in a markdown note *why* it was removed (too little information to reliably impute).


In [ ]:
high_missing_cols = missing_pct[missing_pct > 70].index.tolist()
print("Columns to drop (>70% missing):", high_missing_cols)

df = df.drop(columns=high_missing_cols)
print("Shape after dropping high-missing columns:", df.shape)


## 8. Data-Type Validation & Correction

**Action items:**
- Fix the text-as-number problems found in Schema Validation (e.g. `"twenty"` → `20`, `"eighty"` → `80`). You can build a small word-to-number mapping dictionary for the values you find, or use a library like `word2number`.
- Convert corrected columns to proper numeric dtypes with `pd.to_numeric()`.
- Clean up categorical/text columns: `.str.strip()` to remove whitespace, `.str.lower()` or `.str.title()` to standardize case, and map inconsistent category spellings to one standard value (e.g. `" male "`, `"MALE"` → `"Male"`).
- Re-run `df.info()` afterward to confirm every column now has the correct dtype.


In [ ]:
# Example: word-to-number mapping for columns like Age, Current_GPA, Attendance_Percent, Credits_Registered
word_to_num = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20,
    "eighty": 80,
    # TODO: extend this mapping based on the actual text values you found
}

def clean_numeric(series):
    return series.apply(lambda x: word_to_num.get(str(x).strip().lower(), x))

# TODO: apply clean_numeric() to the specific numeric columns that had text values, then pd.to_numeric()


In [ ]:
# Example: standardizing a categorical column such as Gender
# df['Gender'] = df['Gender'].str.strip().str.lower().replace({
#     'male': 'Male', 'female': 'Female'
# })

# TODO: repeat this pattern for every messy categorical column (e.g. Scholarship_Status, Enrollment_Status)


In [ ]:
df.info()


## 9. Outlier Detection

**Action items:**
- For each continuous numeric column (e.g. GPA, Attendance_Percent), use the IQR method or boxplots to detect outliers.
- Visualize with `seaborn.boxplot()` for the key continuous columns.
- For each outlier group found, make and document a **justified decision**: cap it, correct it (if it's clearly a data-entry error), or leave it if it's a plausible extreme value. Avoid blindly deleting every outlier — explain your reasoning.


In [ ]:
import seaborn as sns

numeric_cols = df.select_dtypes(include=[np.number]).columns
fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(6, 3*len(numeric_cols)))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return series[(series < lower) | (series > upper)]

for col in numeric_cols:
    outliers = iqr_outliers(df[col].dropna())
    print(col, "-> outliers found:", len(outliers))

# TODO: for each column with outliers, decide and apply: cap / correct / retain, and explain why in markdown


## 10. Missing-Value Imputation

**Action items:**
- Choose an imputation strategy per variable type and **explain why** for each:
  - Nominal columns → mode or an explicit `"Unknown"` category.
  - Ordinal columns → mode, or median of the encoded order.
  - Discrete columns → median or group-based median (e.g. median by academic year).
  - Continuous columns → mean/median, or group-based mean/median (e.g. mean GPA by department).
- Use `df[col].fillna(...)`, being careful to **preserve row order** (don't sort/reset in a way that shuffles rows).
- After imputing, re-check `df.isnull().sum()` to confirm no missing values remain (except any column you deliberately decided to leave, with justification).


In [ ]:
# Example patterns — adapt column names to your dataset

# Nominal: mode or 'Unknown'
# df['Scholarship_Status'] = df['Scholarship_Status'].fillna(df['Scholarship_Status'].mode()[0])

# Discrete/Continuous: median, or group-based median
# df['Current_GPA'] = df.groupby('Department')['Current_GPA'].transform(lambda x: x.fillna(x.median()))

# TODO: apply an explicit, explained strategy to every remaining column with missing values


In [ ]:
df.isnull().sum().sort_values(ascending=False)


## 11. Business-Rule Validation

**Action items:** check and correct rows that violate logical business rules, e.g.:
- `Credits_Earned > Credits_Registered` (impossible)
- `Failed_Courses < 0`
- `Backlog_Courses < 0`
- `Assignment_Completion_Percent > 100`

For each rule: count how many rows violate it, decide how to fix (cap, correct, or flag/remove), and apply the fix.


In [ ]:
violations = {
    'Credits_Earned > Credits_Registered': (df['Credits_Earned'] > df['Credits_Registered']).sum(),
    'Failed_Courses < 0': (df['Failed_Courses'] < 0).sum(),
    'Backlog_Courses < 0': (df['Backlog_Courses'] < 0).sum(),
    'Assignment_Completion_Percent > 100': (df['Assignment_Completion_Percent'] > 100).sum(),
}
violations


In [ ]:
# TODO: fix each violation, e.g.:
# df.loc[df['Credits_Earned'] > df['Credits_Registered'], 'Credits_Earned'] = df['Credits_Registered']
# df.loc[df['Failed_Courses'] < 0, 'Failed_Courses'] = 0
# df.loc[df['Backlog_Courses'] < 0, 'Backlog_Courses'] = 0
# df.loc[df['Assignment_Completion_Percent'] > 100, 'Assignment_Completion_Percent'] = 100


## 12. Consistency Validation

**Action items:**
- Check for logically contradictory combinations, e.g. `Academic_Risk_Level = Low` while `Current_GPA = 1.2` and `Attendance_Percent = 52` (a low GPA + low attendance shouldn't be labeled "Low risk").
- Define a simple rule (e.g. based on GPA/attendance thresholds) to flag contradictory rows.
- Decide how to resolve them (recompute the risk label, or flag for manual review) and document your reasoning.


In [ ]:
# Example: flag rows where Academic_Risk_Level looks inconsistent with GPA and Attendance
inconsistent = df[(df['Academic_Risk_Level'] == 'Low') &
                   ((df['Current_GPA'] < 2.0) | (df['Attendance_Percent'] < 60))]
print("Inconsistent rows found:", len(inconsistent))
inconsistent.head()


In [ ]:
# TODO: decide on a resolution rule and apply it, e.g. recompute Academic_Risk_Level
# based on GPA/attendance thresholds, then explain your thresholds in markdown.


## 13. Final Quality Checks

**Action items:**
- Re-run `df.info()`, `df.isnull().sum()`, and `df.duplicated().sum()` to confirm the dataset is now clean.
- Re-check the business rules from Step 11 to confirm zero violations remain.
- Re-check value ranges (e.g. GPA between 0–4, Attendance 0–100%) against the data dictionary.
- Write a short markdown summary confirming the dataset meets production-quality requirements.


In [ ]:
print("Nulls remaining:\n", df.isnull().sum().sum())
print("Duplicates remaining:", df.duplicated().sum())
print(df.describe())


## 14. Save Clean Dataset

**Action items:**
- Export the cleaned DataFrame to a new CSV file, e.g. `university_student_success_production.csv`.
- Use `index=False` so pandas doesn't add an extra unnamed index column.


In [ ]:
df.to_csv("university_student_success_production.csv", index=False)
print("Saved cleaned dataset:", df.shape)


## 15. Data Quality Report / Audit Log (Optional but recommended)

**Action items:**
- Summarize, in a markdown table or short report, every change you made: rows removed (duplicates, blanks), columns dropped, values imputed (and how), outliers handled, business-rule and consistency fixes.
- Include before/after row and column counts.
- This audit log strengthens your submission even though it's optional — it demonstrates you understand *why*, not just *how*.


**Data Quality / Audit Summary** *(fill in with your actual numbers)*

| Step | Issue | Rows/Cols Affected | Action Taken |
|---|---|---|---|
| Duplicate removal | Exact duplicate rows | *(n)* | Dropped |
| Blank rows | Fully blank rows | *(n)* | Dropped |
| High-missing column | >70% missing | *(column name)* | Dropped |
| Data-type errors | Text-as-number values | *(columns)* | Converted |
| Outliers | *(columns)* | *(n)* | *(capped/corrected/retained)* |
| Missing-value imputation | *(columns)* | *(n)* | *(method used)* |
| Business-rule violations | *(rules)* | *(n)* | *(corrected)* |
| Consistency issues | Risk-level vs GPA/Attendance | *(n)* | *(resolved how)* |

**Final shape:** *(rows)* rows × *(columns)* columns


## Conclusion

**Action items:**
- Write 3–5 sentences reflecting on the overall pipeline: what was the biggest data-quality problem, how confident are you the dataset is now production-ready, and what you'd check next before using it to train a machine learning model.
